# Personal AI Hub

Route every query to the best AI model automatically — Claude, Gemini, or GPT.
Pay only for what you use instead of fixed monthly subscriptions.

| Query Type | Best Model | Why |
|---|---|---|
| Upload a PDF / 10-K | Claude Sonnet | Best long-context reading |
| Real-time / news / prices | Gemini Flash | Live Google Search grounding |
| Finance analysis (deep) | Claude Sonnet | Careful with financial details |
| Code writing / debugging | GPT-latest | Strong code model |
| General Q&A | GPT-latest | Strong general-purpose model |

**Setup:** All API keys are already in `screeners/.env`. Just run all cells and open the Gradio URL.

## Cell 1: Install & Imports

In [ ]:
import importlib, subprocess, sys

# Auto-install any missing packages (same pattern as Dual_AI_Stock_Analyzer)
PACKAGES = {
    'anthropic':    'anthropic',
    'google.genai': 'google-genai',
    'openai':       'openai',
    'gradio':       'gradio',
    'dotenv':       'python-dotenv',
    'fitz':         'PyMuPDF',
}

for import_name, pkg_name in PACKAGES.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f'Installing {pkg_name}...')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', pkg_name, '-q'],
            capture_output=True
        )

import anthropic
import openai as openai_lib
from google import genai
from google.genai import types
import gradio as gr
from dotenv import load_dotenv
import os
import fitz  # PyMuPDF — reads uploaded PDF files

print('All packages ready.')

## Cell 2: Configuration

All keys are already in `screeners/.env`. You can also paste them directly below as a fallback.

In [ ]:
# ── API Keys ──────────────────────────────────────────────────────────────────
# Option 1 (VS Code): Keys in screeners/.env  ← all already there
# Option 2 (Colab):   Add in the 🔑 Secrets panel
# Option 3 (fallback): Paste directly below
ANTHROPIC_KEY_DIRECT = ''   # ← paste Anthropic key here if needed
GEMINI_KEY_DIRECT    = ''   # ← paste Gemini key here if needed
OPENAI_KEY_DIRECT    = ''   # ← paste OpenAI key here if needed

load_dotenv()

def _get_secret(name, direct_fallback=''):
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name) or direct_fallback

ANTHROPIC_API_KEY = _get_secret('ANTHROPIC_API_KEY', ANTHROPIC_KEY_DIRECT)
GEMINI_API_KEY    = _get_secret('GEMINI_API_KEY',    GEMINI_KEY_DIRECT)
OPENAI_API_KEY    = _get_secret('OPENAI_API_KEY',    OPENAI_KEY_DIRECT)

if not ANTHROPIC_API_KEY: raise ValueError('Anthropic API key missing — add ANTHROPIC_API_KEY to .env or paste above.')
if not GEMINI_API_KEY:    raise ValueError('Gemini API key missing — add GEMINI_API_KEY to .env or paste above.')
if not OPENAI_API_KEY:    raise ValueError('OpenAI API key missing — add OPENAI_API_KEY to .env or paste above.')

# ── API Clients ───────────────────────────────────────────────────────────────
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
openai_client = openai_lib.OpenAI(api_key=OPENAI_API_KEY)

# ── GPT Model ─────────────────────────────────────────────────────────────────
# Change this ONE line when a new GPT model is released (e.g. 'gpt-5', 'o3')
GPT_MODEL_ID = 'gpt-4.5-preview'  # ← update here to upgrade

# ── Model Roster + Pricing ────────────────────────────────────────────────────
# Pricing: USD per 1 million tokens (input / output)
# Verify current pricing at: platform.openai.com/pricing  &  ai.google.dev/pricing
MODELS = {
    'GPT-latest':    {'id': GPT_MODEL_ID,                'provider': 'openai',  'in': 75.0,  'out': 150.0},
    'Claude Sonnet': {'id': 'claude-sonnet-4-6',         'provider': 'claude',  'in': 3.0,   'out': 15.0},
    'Claude Haiku':  {'id': 'claude-haiku-4-5-20251001', 'provider': 'claude',  'in': 0.25,  'out': 1.25},
    'Gemini Flash':  {'id': 'gemini-2.5-flash',          'provider': 'gemini',  'in': 0.15,  'out': 0.60},
    'Gemini Pro':    {'id': 'gemini-2.0-pro-exp',        'provider': 'gemini',  'in': 2.50,  'out': 10.0},
}

# ── System Prompts ────────────────────────────────────────────────────────────
# These set the AI's persona for each context. Edit freely.
SYSTEM_PROMPTS = {
    'General': (
        'You are a helpful, accurate AI assistant. Answer clearly and concisely.'
    ),
    'Finance Research': (
        'You are an expert financial analyst. Provide thorough, data-driven analysis. '
        'Always note data limitations, cite sources when available, and never make '
        'explicit buy/sell recommendations — focus on analysis for decision-making.'
    ),
    'Code Helper': (
        'You are an expert Python programmer. Write clean, well-commented code. '
        'Explain your reasoning. Prefer pandas, numpy, and plotly for data work.'
    ),
}

# ── Session stats — reset by re-running this cell ─────────────────────────────
session_stats = {'total_cost': 0.0, 'calls': 0, '_last_display': 'No calls yet'}

print('Configuration loaded.')
print(f'  Claude  : claude-sonnet-4-6')
print(f'  Gemini  : gemini-2.5-flash / gemini-2.0-pro-exp')
print(f'  GPT     : {GPT_MODEL_ID}')

## Cell 3: Core Functions

- **Routing** — picks the best model based on query content
- **PDF extraction** — reads uploaded documents via PyMuPDF
- **API calls** — one function per provider (Claude, Gemini, OpenAI)
- **Cost tracking** — calculates spend in USD from token counts

In [ ]:
# ── Routing ───────────────────────────────────────────────────────────────────

def auto_detect_model(message, has_file):
    '''Pick the best model for the task. Returns a key from the MODELS dict.'''

    # File upload → Claude (unique: best long-context reading & instruction following)
    if has_file:
        return 'Claude Sonnet'

    msg = message.lower()

    # Real-time / web data → Gemini Flash (unique: live Google Search grounding)
    real_time_signals = [
        'today', 'right now', 'this week', 'latest news', 'current price',
        'what happened', 'recently', 'just announced', 'breaking',
        'market today', 'stock today', 'what is the price',
    ]
    if any(k in msg for k in real_time_signals):
        return 'Gemini Flash'

    # Finance deep-analysis → Claude (nuanced, careful with financial details)
    finance_signals = [
        '10-k', '10k', 'sec filing', 'annual report', 'balance sheet',
        'income statement', 'cash flow', 'earnings call', 'dcf', 'valuation',
        'compare stocks', 'financial analysis', 'debt-to-equity', 'p/e ratio',
        'fundamental analysis', 'earnings per share',
    ]
    if any(k in msg for k in finance_signals):
        return 'Claude Sonnet'

    # Code / programming → GPT (strong code model)
    code_signals = [
        'write a function', 'write code', 'write a script', 'write a python',
        'debug this', 'error in my code', 'refactor', 'def ',
        'how do i implement', 'write me a program',
    ]
    if any(k in msg for k in code_signals):
        return 'GPT-latest'

    # Default → GPT (strong general-purpose model)
    return 'GPT-latest'


def resolve_model(model_choice, message, has_file):
    '''Return the actual model key to use (handles Auto selection).'''
    if model_choice == 'Auto':
        return auto_detect_model(message, has_file)
    return model_choice


# ── File Handling ─────────────────────────────────────────────────────────────

def extract_file_text(file_path):
    '''Extract text from an uploaded PDF. Returns None if no file provided.'''
    if file_path is None:
        return None
    if file_path.lower().endswith('.pdf'):
        try:
            doc = fitz.open(file_path)
            text = ''.join(page.get_text() for page in doc)
            doc.close()
            # Cap at 80K chars to avoid token overflow on very large documents
            if len(text) > 80_000:
                text = text[:80_000] + '\n\n[Document truncated at 80,000 characters]'
            return text
        except Exception as e:
            return f'[Error reading PDF: {e}]'
    return None


# ── Cost Calculation ──────────────────────────────────────────────────────────

def calculate_cost(model_key, in_tokens, out_tokens):
    '''Return cost in USD for a single API call based on token counts.'''
    m = MODELS[model_key]
    return (in_tokens * m['in'] + out_tokens * m['out']) / 1_000_000


# ── API Calls — one function per provider ─────────────────────────────────────

def call_claude(model_id, messages, system):
    '''Call Anthropic Claude. Returns (text, input_tokens, output_tokens).'''
    r = claude_client.messages.create(
        model=model_id,
        max_tokens=4096,
        system=system,
        messages=messages,
    )
    return r.content[0].text, r.usage.input_tokens, r.usage.output_tokens


def call_gemini(model_id, messages, system, use_search=False):
    '''Call Google Gemini. Returns (text, input_tokens, output_tokens).'''
    # Convert OpenAI-style message list to Gemini contents format
    contents = []
    for h in messages:
        role = 'user' if h['role'] == 'user' else 'model'
        contents.append(types.Content(
            role=role,
            parts=[types.Part(text=h['content'])]
        ))

    config_kwargs = {'temperature': 0.3, 'system_instruction': system}
    if use_search:
        # Enable live Google Search grounding (Gemini Flash only)
        config_kwargs['tools'] = [types.Tool(google_search=types.GoogleSearch())]

    r = gemini_client.models.generate_content(
        model=model_id,
        contents=contents,
        config=types.GenerateContentConfig(**config_kwargs),
    )

    in_tok  = r.usage_metadata.prompt_token_count or 0
    out_tok = r.usage_metadata.candidates_token_count or 0
    return r.text, in_tok, out_tok


def call_openai(model_id, messages, system):
    '''Call OpenAI GPT. Returns (text, input_tokens, output_tokens).'''
    full_messages = [{'role': 'system', 'content': system}] + messages
    r = openai_client.chat.completions.create(
        model=model_id,
        messages=full_messages,
        max_tokens=4096,
    )
    return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens


# ── Master Router ─────────────────────────────────────────────────────────────

def route_and_call(message, model_choice, history, file_path, context):
    '''
    Route to the correct provider and return:
        (response_text, cost_usd, resolved_model_name)

    history: list of {'role': 'user'/'assistant', 'content': '...'} dicts
    '''
    has_file = file_path is not None
    resolved = resolve_model(model_choice, message, has_file)
    m = MODELS[resolved]
    system = SYSTEM_PROMPTS.get(context, SYSTEM_PROMPTS['General'])

    # Prepend uploaded file content before the user's question
    full_message = message
    file_text = extract_file_text(file_path)
    if file_text:
        full_message = (
            '[Uploaded document:]\n\n' + file_text +
            '\n\n[Question:]\n' + message
        )

    messages = history + [{'role': 'user', 'content': full_message}]

    provider = m['provider']
    if provider == 'claude':
        text, in_tok, out_tok = call_claude(m['id'], messages, system)
    elif provider == 'gemini':
        use_search = (resolved == 'Gemini Flash')  # only Flash has web grounding
        text, in_tok, out_tok = call_gemini(m['id'], messages, system, use_search)
    elif provider == 'openai':
        text, in_tok, out_tok = call_openai(m['id'], messages, system)
    else:
        raise ValueError(f'Unknown provider: {provider}')

    cost = calculate_cost(resolved, in_tok, out_tok)
    return text, cost, resolved


print('Core functions loaded.')

## Cell 4: Gradio Interface

Builds the chat UI. Run **Cell 5** after this to open the app.

In [ ]:
def format_cost(last_cost, resolved):
    '''Format the session cost tracker display string.'''
    total = session_stats['total_cost']
    calls = session_stats['calls']
    return (
        f'Last call: ${last_cost:.4f} via {resolved}\n'
        f'Session total: ${total:.4f} ({calls} calls)'
    )


def chat(message, history, model_choice, context, file_obj):
    '''Main handler — called by Gradio each time the user sends a message.'''
    if not message.strip():
        return history, session_stats['_last_display'], ''

    file_path = file_obj if file_obj else None

    # Convert Gradio messages format to API-compatible list
    api_history = [{'role': h['role'], 'content': h['content']} for h in history]

    try:
        response_text, cost, resolved = route_and_call(
            message, model_choice, api_history, file_path, context
        )
        session_stats['total_cost'] += cost
        session_stats['calls'] += 1
        cost_display = format_cost(cost, resolved)
        session_stats['_last_display'] = cost_display
    except Exception as e:
        response_text = f'Error calling API: {e}'
        cost_display = session_stats['_last_display']

    history = history + [
        {'role': 'user',      'content': message},
        {'role': 'assistant', 'content': response_text},
    ]

    return history, cost_display, ''  # '' clears the input box after sending


def preview_model(message, model_choice):
    '''Show which model will be selected as the user types.'''
    if model_choice == 'Auto':
        detected = auto_detect_model(message or '', False)
        return f'Auto → {detected}'
    return model_choice


with gr.Blocks(title='Personal AI Hub', theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        '# Personal AI Hub\n'
        '**Quality-First AI Router — Claude · Gemini · GPT**'
    )

    # ── Top row: Context + Model selector + Live preview ──────────────────────
    with gr.Row():
        context_dd = gr.Dropdown(
            choices=list(SYSTEM_PROMPTS.keys()),
            value='General',
            label='Context',
            scale=1,
        )
        model_dd = gr.Dropdown(
            choices=['Auto'] + list(MODELS.keys()),
            value='Auto',
            label='Model',
            scale=1,
        )
        will_use = gr.Textbox(
            value='Auto → GPT-latest',
            label='Will use',
            interactive=False,
            scale=2,
        )

    # ── Chat window ───────────────────────────────────────────────────────────
    chatbot = gr.Chatbot(
        height=480,
        show_label=False,
        type='messages',
        placeholder='Ask a finance question, upload a PDF, write some code...',
    )

    # ── File upload + Cost tracker ────────────────────────────────────────────
    with gr.Row():
        file_upload = gr.File(
            label='Upload PDF',
            file_types=['.pdf'],
            scale=2,
        )
        cost_box = gr.Textbox(
            label='Session Cost',
            value='No calls yet',
            interactive=False,
            lines=2,
            scale=2,
        )

    # ── Input row ─────────────────────────────────────────────────────────────
    with gr.Row():
        msg_box = gr.Textbox(
            placeholder='Type your message here...',
            show_label=False,
            scale=5,
            lines=2,
        )
        send_btn = gr.Button('Send', variant='primary', scale=1, min_width=80)

    clear_btn = gr.Button('Clear conversation', variant='secondary')

    # ── Wire up events ────────────────────────────────────────────────────────

    # Live model preview updates as the user types
    msg_box.change(fn=preview_model, inputs=[msg_box, model_dd], outputs=[will_use])
    model_dd.change(fn=preview_model, inputs=[msg_box, model_dd], outputs=[will_use])

    # Submit on button click or Enter key
    send_btn.click(
        fn=chat,
        inputs=[msg_box, chatbot, model_dd, context_dd, file_upload],
        outputs=[chatbot, cost_box, msg_box],
    )
    msg_box.submit(
        fn=chat,
        inputs=[msg_box, chatbot, model_dd, context_dd, file_upload],
        outputs=[chatbot, cost_box, msg_box],
    )

    # Clear conversation history (session cost total continues)
    clear_btn.click(
        fn=lambda: ([], 'Conversation cleared. Session cost continues.'),
        inputs=[],
        outputs=[chatbot, cost_box],
    )

print('Gradio UI built. Run Cell 5 to launch.')

## Cell 5: Launch

Opens the hub in your browser at `http://localhost:7860`.  
Login uses `GRADIO_USERNAME` / `GRADIO_PASSWORD` from `.env` (defaults: `user` / `hub123`).  
**To stop:** interrupt the kernel (square stop button in Jupyter).

In [ ]:
username = os.environ.get('GRADIO_USERNAME', 'user')
password = os.environ.get('GRADIO_PASSWORD', 'hub123')

# Colab needs share=True to expose a public URL (it's a cloud VM, not localhost).
# VS Code uses share=False so it stays private on your machine.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Launching Personal AI Hub...')
if IN_COLAB:
    print('  Colab detected — a public Gradio URL will be generated.')
    print(f'  Login: {username} / {password}')
else:
    print(f'  Login: {username} / {password}')
    print('  URL:   http://localhost:7860')

demo.launch(
    server_name='0.0.0.0',
    share=IN_COLAB,      # True in Colab (required), False in VS Code (stays local)
    auth=(username, password),
    debug=False,
)